## Backpropagation Through Time

BPTT - способ обучения рекурентной нейронной сети. В случае с RNN-ками мы работаем с набором предсказаний и каждая ошибка должна корректировать не только прямой слой но и предыдущие ошибки

Один прогон рекуррентной сети = обработанная последовательность из T элементов

Дадим формальное описание процесса применения (forward pass). Рассмотрим стандартную рекуррентную сеть, у нее
- два входа:
    - внутренний state $h_{t-1}$
    - внешний input $x_t$
- два выхода:
    - внутренний state $h_{t}$
    - внешний output $y_{t}$


<img src="img/rnn.png" width=200>

Для каждого $t \in 1 ... T$ входы и выходы связаны такими соотношениями

$
\begin{cases}
      y_{t} = w_{y} \cdot h_{t-1} \\
      h_{t} = w_{h} \cdot h_{t-1} + w_{x} \cdot x
\end{cases}
$



#### Формула для $h_t$
Каждый выход $h_{t}$ зависит от всех предыдущих состояний $h_{t-1}, h_{t-2} ... h_{0}$<br>
Значение любого можно развернуть в сумму Например для шага s:

$
h_{s} = w_{h}h_{s-1} + w_x x_{s}
$

$
h_{s} = w_{h}(w_{h}h_{s-2}+w_{x}x_{s-1}) + w_{x}x_{s} 
$

$
h_{s} = w_{h}(w_{h} (w_{h} h_{s-3} + w_{x} x_{s-2})+w_{x}x_{s-1}) + w_{x}x_{s}
$

И так можно развернуть до самого первого слагаемого:

$
h_{s} = w_h^{s-1} \cdot w_x \cdot x_{s-2} + w_h^{s-2} \cdot w_x \cdot x_{s-3} + ... + w_h \cdot w_x + h_0 \cdot w_{h} 
$

Удобно свернуть в сумму. Итого у нас появляется две формцы записи для каждого выхода - через соседний предыдущий выход и через все предыдущие выходы

$
\begin{cases}
      h_{s} = w_{h} \cdot h_{s-1} + w_{x} \cdot x \\
      h_{s}=w^t \cdot h_0 + \sum_{k=1}^{t} w_{h}^{T-k} \cdot h_{k-1}      
\end{cases}
$

#### Формула для $h_t'$
Выпишем аналогично для производных. Продифференцировать первыое выражение

$$\frac{\partial h_{s}}{\partial w_{h}} = \frac{\partial}{\partial w_h} \bigg[ w_{h} \cdot h_{s-1} + w_{x} x \bigg] = \frac{\partial}{\partial w_h} \bigg[ w_{h} \cdot h_{s-1}\bigg]$$

Орбащаем внимание что поскольку $w$ здесь не константа, а собвстенно то, по чему мы дифференцируем, дифференциоровать надо оба слагаемых $(x \cdot f(x))' = f + x \cdot f'$

Итого получаем рекурренное выражение для градиентов

$$\frac{\partial h_{s}}{\partial w_{h}} = \frac{\partial h_{s-1}}{\partial w_h} + w_{h}$$

Это выражение можно развернуть до любого щага $h_S$

$$
\frac{\partial h_{s}}{\partial w_{h}} = \sum_{k=1}^s w_{h}^{s-k} h_{k-1}
$$

## BPTT

Пусть задана функция потерь. Она аддитивна и суммируется из функций потерь всех внешних выходов

$L=\sum_{k=1}^T l(y_k,\hat{y_k})$

Можно относительно другого выхода посчитать производную. Она будет совсем простой:

$$
\frac{\partial h_s}{\partial h_k} = w_{h}^{\,s-k}
$$

Ткеперь производную по параметру

$$
\frac{\partial \ell}{\partial h_k} = \frac{\partial \ell}{\partial y_s} \frac{\partial y_s}{\partial h_s} \frac{\partial h_s}{\partial h_k}
= \ell'(y_s)\, w_{hy}\, w_{hh}^{\,s-k}
$$

$$
\frac{\partial \ell}{\partial w_{hy}} = \ell'(y_s)\, h_s
$$

$$
\frac{\partial \ell}{\partial w_{hh}} 
= \sum_{k=1}^{s} 
   \big(\ell'(y_s)\, w_{hy}\, w_{hh}^{\,s-k}\big)\, h_{k-1}
$$

$$
\frac{\partial \ell}{\partial w_{xh}} 
= \sum_{k=1}^{s} 
   \big(\ell'(y_s)\, w_{hy}\, w_{hh}^{\,s-k}\big)\, x_k
$$

